## Imports

Libraries you will need throughout this notebook:

- `datasets` — load MRPC
- `transformers` — tokenizer, model, `Trainer`, `TrainingArguments`
- `evaluate` — compute accuracy and F1
- `numpy` — argmax
- `torch` — Part 2 (custom training loop)
- `accelerate` — Part 2 (distributed-ready training)


In [1]:
import numpy as np
import evaluate
import torch

from datasets import load_dataset
from torch import accelerator
from torch.ao.nn.quantized import reference
from transformers import AutoTokenizer, DataCollatorWithPadding, TrainingArguments, AutoModelForSequenceClassification, Trainer, get_scheduler
from accelerate import Accelerator
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm



/home/v/Desktop/Projects/HuggingFace/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and inspect the dataset

Load MRPC with `load_dataset("glue", "mrpc")`.  
The dataset has three splits: `train` (3 668 pairs), `validation` (408), `test` (1 725).

Each example has four columns: `sentence1`, `sentence2`, `label`, `idx`.  
Labels: `0` = not equivalent, `1` = equivalent (paraphrase).

In [2]:
raw_datasets = load_dataset("glue", "mrpc")

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


## Tokenize sentence pairs

Checkpoint: `bert-base-uncased`

MRPC has *pairs* of sentences. The tokenizer handles pairs directly:  
```
tokenizer(sentence1, sentence2, truncation=True)
```
It adds `[CLS]`, `[SEP]` tokens and the `token_type_ids` field to tell BERT which sentence is which.

Write a `tokenize_function` that takes a batch and tokenizes the pair.


In [3]:
checkpoint = "bert-base-uncased" #Pretrained model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_batch(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)

print(tokenized_datasets["train"])

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3668
})


## DataCollatorWithPadding

A **data collator** runs on each batch just before it reaches the model.  
Its job: take a list of variable-length examples and pad them all to the same length (the longest in the batch).

`DataCollatorWithPadding(tokenizer)` is the standard choice — it knows which token to use for padding and whether to pad left or right.

In [4]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Load the model

Load `bert-base-uncased`.


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## compute_metrics

The `Trainer` calls `compute_metrics` after each evaluation run.  
It receives an `EvalPrediction` with:
- `.predictions` — raw **logits** from the model, shape `(n_examples, n_labels)` = `(408, 2)`
- `.label_ids` — ground-truth labels, shape `(408,)`

---

> **Why logits and not probabilities?**  
> The model outputs raw, unnormalized scores — one per class.  
> To find the predicted class you just need the *highest* score — `argmax` does that without calling `softmax`.  
> Example: logits `[-0.3, 1.8]` → argmax = `1` (paraphrase). No softmax needed.
>
> **Why `axis=-1`?**  
> `.predictions` is 2D: rows are examples, columns are classes.  
> `axis=-1` = last axis = collapse across the classes dimension. Equivalent to `axis=1` here, but more robust.

In [6]:
def compute_metrics(eval_predictions):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_predictions
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## TrainingArguments

Create a `TrainingArguments` object:

| Parameter | Value | Why |
|---|---|---|
| `output_dir` | `"test-trainer"` | Checkpoint directory |
| `eval_strategy` | `"epoch"` | Evaluate after each epoch |
| `save_strategy` | `"epoch"` | Save checkpoint after each epoch |
| `learning_rate` | `2e-5` | Standard for BERT-family fine-tuning |
| `per_device_train_batch_size` | `16` | |
| `per_device_eval_batch_size` | `16` | |
| `num_train_epochs` | `3` | |
| `weight_decay` | `0.01` | L2 regularization |
| `load_best_model_at_end` | `True` | Restore best checkpoint when done |

In [7]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch", num_train_epochs=5)

## Trainer

Instantiate `Trainer` with:
- `model`, `args`
- `train_dataset`, `eval_dataset`
- `data_collator`
- `processing_class` (the tokenizer — replaces the deprecated `tokenizer=` parameter)
- `compute_metrics`

Then call `.train()`.


In [8]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.490655,0.791667,0.866562
2,0.542500,0.523427,0.833333,0.886288
3,0.308500,0.645564,0.852941,0.892086
4,0.145900,0.823050,0.850490,0.893913
5,0.048100,0.848685,0.855392,0.897747


TrainOutput(global_step=2295, training_loss=0.2314602696038539, metrics={'train_runtime': 97.3924, 'train_samples_per_second': 188.31, 'train_steps_per_second': 23.564, 'total_flos': 675891190117440.0, 'train_loss': 0.2314602696038539, 'epoch': 5.0})

## Evaluation

Use `trainer.predict(tokenized_datasets["validation"])` to get predictions.  
The result is a `PredictionOutput` with `.predictions`, `.label_ids`, and `.metrics`.

**Your task:**
1. Run `trainer.predict` on the validation split.
2. Convert logits to class indices manually with `np.argmax`.
3. Compute metrics manually with `evaluate.load("glue", "mrpc").compute(...)` to verify they match the logged values.
4. Print 5 example predictions alongside the original sentence pairs.

In [9]:
prediction_output = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(prediction_output.predictions, axis=-1)
metric = evaluate.load("glue", "mrpc").compute(predictions=predictions, references=prediction_output.label_ids)

print(tokenized_datasets["validation"][0]["sentence1"])
print(tokenized_datasets["validation"][0]["sentence2"])
print("Dataset_label: ",tokenized_datasets["validation"][0]["label"])
print("Prediction label: ", predictions[0])

He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
" The foodservice pie business does not fit our long-term growth strategy .
Dataset_label:  1
Prediction label:  1


## Why write a custom loop?

The `Trainer` covers 95 % of use cases, but sometimes you need full control:
- Custom loss functions
- Non-standard optimization schedules
- Gradient manipulation (clipping, accumulation by hand)
- Debugging what exactly happens at each step

The custom loop also makes the Trainer less of a black box — once you write it by hand, you understand what the Trainer is doing for you.

## Prepare the dataloaders

The `Trainer` creates `DataLoader`s internally. Here you create them yourself.

Before creating the dataloaders, the tokenized dataset needs three manual steps that the Trainer did automatically:
1. Remove columns the model doesn't accept (`sentence1`, `sentence2`, `idx`)
2. Rename `label` → `labels` (the model expects the argument named `labels`)
3. Set format to `torch` (returns tensors instead of Python lists)

Then create:
- `train_dataloader` with `shuffle=True`, `batch_size=8`, `collate_fn=data_collator`
- `eval_dataloader` with `batch_size=8`, `collate_fn=data_collator`

Shuffling prevents the model from memorizing the order of examples — important for training.

In [10]:
def prepare_dataset(ds):
    ds = ds.remove_columns(["sentence1", "sentence2", "idx"])
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")
    return ds


prepared_ds = prepare_dataset(tokenized_datasets)

train_dataloader = DataLoader(
    prepared_ds["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    prepared_ds["validation"], batch_size=8, collate_fn=data_collator
)

## Optimizer and learning rate scheduler

The Trainer uses **AdamW** and a **linear decay** schedule by default. Replicate both:

In [11]:
optimizer = AdamW(model.parameters(), lr=3e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

## Move the model to GPU

The Trainer handles device placement automatically. In a manual loop, you do it yourself:

In [12]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## The training loop

Write the training loop following this sequence **for each batch**:

```
1. Move batch to device
2. Forward pass:  outputs = model(**batch)
3. Extract loss:  loss = outputs.loss
4. Backward pass: loss.backward()
5. Optimizer step
6. Scheduler step
7. Zero gradients:  optimizer.zero_grad()
8. Update progress bar
```

Use `tqdm` for the progress bar.

In [14]:
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)


  0%|          | 0/1377 [01:34<?, ?it/s]

100%|█████████▉| 1376/1377 [00:57<00:00, 24.62it/s]

## The evaluation loop

Write the evaluation loop. Key differences from the training loop:
- `model.eval()` — disables dropout and batch norm (different behavior at test time)
- `with torch.no_grad()` — disables gradient computation (saves memory and compute)
- `metric.add_batch()` — accumulates predictions across batches, then call `metric.compute()` once at the end

In [19]:
metric = evaluate.load("glue", "mrpc")

model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8504901960784313, 'f1': 0.897133220910624}

## Accelerate — distributed training with minimal changes

The training loop above runs on a single CPU or GPU. `accelerate` makes it run on multiple GPUs or TPUs with four changes:

**Inside the loop:**
- Replace `loss.backward()` with `accelerator.backward(loss)`
- Remove all manual .to(device) calls. Accelerate handles device placement.

**Why does this matter even on a single GPU?**
The same code runs unchanged on 1 GPU, 8 GPUs, a TPU pod, or a CPU — no `if torch.cuda.is_available()` branches needed.
The Trainer uses Accelerate internally, which is why it works on any hardware without code changes.

In [20]:
accelerator = Accelerator()

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

1530it [11:09,  2.28it/s]00:00<?, ?it/s]
100%|█████████▉| 1376/1377 [01:03<00:00, 21.96it/s]

## What are learning curves?

Learning curves are plots of **loss** and **accuracy** (or any metric) over training steps or epochs.  
They tell you whether the model is learning, over-learning, or not learning at all.

You get them for free from the Trainer when `eval_strategy="epoch"` — the metrics are logged after each epoch.  
For richer, interactive plots, integrate [Weights & Biases](https://wandb.ai): add `report_to="wandb"` to `TrainingArguments`.

**Your task:** after training in Section 3.3, plot the `eval_loss`, `eval_accuracy`, and `eval_f1` across the 3 epochs.  
The metrics are in `trainer.state.log_history`. Print it to see the raw values first.

## The three patterns to recognize

### 1. Healthy training
- Training loss decreases steadily
- Validation loss follows closely (small gap between train and val)
- Validation accuracy increases and converges
- Accuracy curve looks "steppy" — this is normal (see note below)

### 2. Overfitting
- Training loss keeps decreasing
- Validation loss *increases* after a point (the curves diverge)
- Large gap between training and validation accuracy

**Solutions:** early stopping, weight decay, dropout, more data

### 3. Underfitting
- Both training and validation loss remain high
- Performance plateaus early in training

**Solutions:** train longer, higher learning rate, larger model, better data

---

## Key takeaways — Chapter 3

| Concept | What to remember |
|---|---|
| Dynamic padding | Pad per batch via `DataCollatorWithPadding`, not globally — avoids wasted compute |
| Logits → predictions | `np.argmax(logits, axis=-1)` — no softmax needed for classification |
| Classification head | Pre-trained body + randomly initialized linear head → fine-tune both |
| `weight_decay` | L2 regularization via AdamW — decoupled from the adaptive learning rates |
| `load_best_model_at_end` | Restores best checkpoint — pair with `save_strategy="epoch"` |
| Training loop order | forward → backward → optimizer step → scheduler step → zero gradients |
| `model.eval()` + `no_grad()` | Always use both during evaluation — correct behavior + memory savings |
| Accelerate | 4 changes to make any training loop run on any hardware |
| Healthy curves | Train and val loss both decrease, small gap between them |
| Overfitting signal | Val loss increases while train loss decreases |

---

> **Coming in Chapter 11:** PEFT, LoRA, SFT (Supervised Fine-Tuning), DPO, and chat templates.